In [1]:
import numpy as np
import pandas as pd
import random
import time

In [2]:
teams = [
    'crd','atl','rav','buf','car','chi','cin','cle',
    'dal','den','det','gnb','htx','clt','jax','kan',
    'sdg','ram','rai','mia','min','nwe','nor','nyg',
    'nyj','phi','pit','sea','sfo','tam','oti','was'
]

print(f'number of teams={len(teams)}')

number of teams=32


In [3]:
rename_dict = {
    'Unnamed: 5':'Home', 'Rslt':'Win', 'Pts':'Tm_Pts', 'PtsO':'Opp_Pts',
    'Cmp':'pCmp', 'Att':'pAtt', 'Cmp%':'pCmp%', 'Yds':'pYds',
    'TD':'pTD', 'Y/A':'pY/A', 'AY/A':'pAY/A', 'Rate':'pRate',
    'Yds.1':'SkYds', 'Att.1':'rAtt', 'Yds.2':'rYds', 'TD.1':'rTD',
    'Y/A.1':'rY/A', 'Yds.3':'PntYds', 'Pass':'fdPass', 'Rsh':'fdRush',
    'Pen':'fdPen', 'Pen.1':'Pen', 'Yds.4':'PenYds'
}

In [6]:
start_time = time.time()

seasons = range(2002, 2015)

nfl_df = pd.DataFrame()

for season in seasons:
    for team in teams:
        url = 'https://www.pro-football-reference.com/teams/' + team + '/' + str(season) + '/gamelog/'
        print(url)

        table_id = 'table_pfr_team-year_game-logs_team-year-regular-season-game-log'
        tm_df = pd.read_html(url, header=1, attrs={'id':table_id})[0]

        # Drop rows where the 'Rk' value is NaN and rename some columns
        tm_df = tm_df.dropna(subset=['Rk'])
        tm_df = tm_df.rename(rename_dict, axis=1)

        # Add 'Tm_' prefix to game stats
        pre_dict = {col:f'Tm_{col}' for col in tm_df.columns[11:]}
        tm_df = tm_df.rename(columns=pre_dict)

        # Get Opponent Stats (from Opponent Game Logs table)
        table_id = 'table_pfr_team-year_game-logs_team-year-regular-season-opponent-game-log'
        opp_df = pd.read_html(url, header=1, attrs={'id':table_id})[0]

        # Drop rows where the 'Rk' value is NaN and rename some columns
        opp_df = opp_df.dropna(subset=['Rk'])
        opp_df = opp_df.rename(rename_dict, axis=1)

        # Add 'Opp_' prefix to game stats
        pre_dict = {col:f'Opp_{col}' for col in opp_df.columns[11:]}
        opp_df = opp_df.rename(columns=pre_dict)

        # Create columns to merge
        cols_to_merge = tm_df.columns[:11].tolist()
        
        # Merge based on the first eleven columns
        merged_df = pd.merge(tm_df, opp_df, on=cols_to_merge)
        
        # Insert Season and Team as new columns
        merged_df.insert(loc=0, column='Season', value=season)
        merged_df.insert(loc=1, column='Team', value=team.upper())
        
        # APPEND TO NFL_DF (aggregate dataframe)
        
        # Concatenate the team gamelog to the aggregate dataframe
        nfl_df = pd.concat([nfl_df, merged_df], ignore_index=True)
        
        # Pause the program to abide by website rules ("The Dude abides" - Jeff Lebowski)
        time.sleep(random.randint(4, 5))

# Get the ending time
end_time = time.time()

# Display the elapsed time
print(f'Elapsed time: {end_time - start_time:1f} seconds')

# Display the aggregate dataframe
print(nfl_df.info())

https://www.pro-football-reference.com/teams/crd/2002/gamelog/
https://www.pro-football-reference.com/teams/atl/2002/gamelog/
https://www.pro-football-reference.com/teams/rav/2002/gamelog/
https://www.pro-football-reference.com/teams/buf/2002/gamelog/
https://www.pro-football-reference.com/teams/car/2002/gamelog/
https://www.pro-football-reference.com/teams/chi/2002/gamelog/
https://www.pro-football-reference.com/teams/cin/2002/gamelog/
https://www.pro-football-reference.com/teams/cle/2002/gamelog/
https://www.pro-football-reference.com/teams/dal/2002/gamelog/
https://www.pro-football-reference.com/teams/den/2002/gamelog/
https://www.pro-football-reference.com/teams/det/2002/gamelog/
https://www.pro-football-reference.com/teams/gnb/2002/gamelog/
https://www.pro-football-reference.com/teams/htx/2002/gamelog/
https://www.pro-football-reference.com/teams/clt/2002/gamelog/
https://www.pro-football-reference.com/teams/jax/2002/gamelog/
https://www.pro-football-reference.com/teams/kan/2002/g

In [7]:
# Clean the data

# Drop the Rk column
nfl_df = nfl_df.drop(columns=['Rk'], axis=1)

# Convert 'Home' column to 1 = Home or 0 = Away
nfl_df['Home'] = np.where(nfl_df['Home'] == '@', 0, 1)

# Convert 'Win' column to 1 = Win or 0 = Loss (or Tie)
nfl_df['Win'] = np.where(nfl_df['Win'] == 'W', 1, 0)

# Convert 'OT' column to 1 = OT or 0 = No OT
nfl_df['OT'] = np.where(nfl_df['OT'] == 'OT', 1, 0)

# Display the info
print(nfl_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6656 entries, 0 to 6655
Data columns (total 86 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Season      6656 non-null   int64  
 1   Team        6656 non-null   object 
 2   Gtm         6656 non-null   float64
 3   Week        6656 non-null   float64
 4   Date        6656 non-null   object 
 5   Day         6656 non-null   object 
 6   Home        6656 non-null   int64  
 7   Opp         6656 non-null   object 
 8   Win         6656 non-null   int64  
 9   Tm_Pts      6656 non-null   int64  
 10  Opp_Pts     6656 non-null   int64  
 11  OT          6656 non-null   int64  
 12  Tm_pCmp     6656 non-null   int64  
 13  Tm_pAtt     6656 non-null   int64  
 14  Tm_pCmp%    6656 non-null   float64
 15  Tm_pYds     6656 non-null   int64  
 16  Tm_pTD      6656 non-null   int64  
 17  Tm_pY/A     6656 non-null   float64
 18  Tm_pAY/A    6656 non-null   float64
 19  Tm_pRate    6656 non-null  

In [8]:
nfl_df.to_csv('nfl_dataset.csv', index=False)